# MLflow GenAI lifecycle — end-to-end demo

Walks the **full MLflow GenAI lifecycle loop** on a tiny
PolicyPal-shape example so the unified story lives in a single
file. Each per-phase notebook covers one phase in depth; the
table below maps each phase to the notebook that exercises it.

| # | Phase    | Per-phase notebook          |
|---|----------|-----------------------------|
| 1 | Author   | c0301, c0701, c0901         |
| 2 | Register | c0301 (prompt registry)     |
| 3 | Trace    | c0701, c0901, c1301         |
| 4 | Evaluate | c1301                       |
| 5 | Optimize | c1101 (prompt promotion)    |
| 6 | Deploy   | c1001                       |
| + | Monitor  | c1401                       |

**Demo task:** a `PolicyClassifierAgent` that reads a one-line
PawShield email and returns one of `claim` / `question` / `appeal`
/ `complaint`. The smallest task that exercises every phase.

For the **production-scale** version (UC function tools, managed
MCP, real ReAct loop), see
`pawshield-notebooks/09-agents/c0901-claimclerk-agent-with-mcp.py`.
For the **rigorous evaluation** version (20-row ground-truth Delta
table, citation_correctness scorer), see
`pawshield-notebooks/13-evaluation/c1301-mlflow-eval-policy-qa.py`.

**Prerequisites:** serverless compute (Databricks default), MLflow
3.x runtime with `mlflow.genai` APIs, the
`databricks-meta-llama-3-1-8b-instruct` pay-per-token endpoint, and
Unity Catalog `genaicert.pawshield` schema (created by the c0001 setup notebook).

In [0]:
# NOTE: %pip and !pip are intercepted by PipMagicOverrides on Serverless,
# which calls spark.conf.set() to register environment metadata. If the
# Spark Connect session has expired (INACTIVITY_TIMEOUT), this will fail.
# Fix: detach & reattach compute to get a fresh session, then re-run.
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install --quiet --force-reinstall "mlflow[databricks]>=3.12,<3.13" databricks-sdk databricks-openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("schema", "pawshield")
dbutils.widgets.text("prompt_name", "policy_classifier_intent")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
prompt_short_name = dbutils.widgets.get("prompt_name")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
prompt_name = f"{catalog}.{schema}.{prompt_short_name}"

In [0]:
print(f"Catalog/schema: {catalog}.{schema}")
print(f"Prompt name:    {prompt_name}")
print(f"LLM endpoint:   {llm_endpoint}")

Catalog/schema: genaicert.pawshield
Prompt name:    genaicert.pawshield.policy_classifier_intent
LLM endpoint:   databricks-meta-llama-3-1-8b-instruct


## 1. Author

A `ResponsesAgent` subclass wrapping a single classify call —
smallest shape that still exercises the full lifecycle. The system
prompt is short on purpose; Phase 2 registers it so it becomes a
versioned artifact, not a magic string. Tools use typed signatures
+ Google-style docstrings so `@mlflow.trace` can label spans.

Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent

In [0]:
# The agent class lives in `policy_classifier_agent.py` — a standalone
# workspace file in this folder. Two reasons it's not inline:
# (1) `mlflow.pyfunc.log_model(python_model=<path>)` needs a real .py
#     file path. Databricks workspace **notebooks** uploaded with
#     `--format SOURCE` have no `.py` on disk; only workspace **files**
#     do. policy_classifier_agent.py was uploaded with `--format AUTO`.
# (2) Parameterisation lives in `model_config` (read in `load_context`),
#     so one class serves multiple deployments — Phase 6 logs the same
#     file with different prompt URIs without touching agent code.
#
# `sys.path.insert(...)` (side effect on the interpreter's import path)
# and `from policy_classifier_agent import ...` (the actual module load)
# are co-located because both must run for the module to become visible.
import mlflow
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

# Patch: databricks-vectorsearch 0.74 renamed VectorSearchIndex → AISearchIndex,
# but databricks-ai-bridge still imports the old name at package init.
import databricks.vector_search.client as _vsc_mod
if not hasattr(_vsc_mod, 'VectorSearchIndex'):
    _vsc_mod.VectorSearchIndex = _vsc_mod.AISearchIndex

import sys
sys.path.insert(0, "/Workspace" + dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0])

from policy_classifier_agent import (
    PolicyClassifierAgent,
    INTENT_LABELS,
    IntentLabel,
    DEFAULT_PROMPT_TEMPLATE,
)
from types import SimpleNamespace

In [0]:
# Build a fresh instance for in-notebook testing (eval + optimize
# phases). Manually invoke `load_context` with the widget-derived
# config so the instance is fully constructed before Phase 3 calls
# `agent.predict(...)`. At serve time MLflow does the same thing
# automatically using the model_config passed to log_model in Phase 6.
agent = PolicyClassifierAgent()
agent.load_context(SimpleNamespace(model_config={
    "prompt_uri":   f"prompts:/{prompt_name}@champion",
    "llm_endpoint": llm_endpoint,
}))

In [0]:
print(f"Authored PolicyClassifierAgent bound to prompt URI: prompts:/{prompt_name}@champion")

Authored PolicyClassifierAgent bound to prompt URI: prompts:/genaicert.pawshield.policy_classifier_intent@champion


## 2. Register the prompt

The Prompt Registry treats prompts as first-class versioned
artifacts. Aliasing `@champion` to a version gives prompt
promotions the same lifecycle controls as model promotions —
roll forward or back by moving the alias, no code change.

Source: https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry
# `register_prompt` creates a new version each call; `set_prompt_alias`
# is the promotion lever. Chains/agents load via `prompts:/<name>@<alias>`.
registered = mlflow.genai.register_prompt(
    name=prompt_name,
    template=DEFAULT_PROMPT_TEMPLATE,
    commit_message="v1 — initial intent-router prompt for PolicyPal",
)
mlflow.genai.set_prompt_alias(
    name=prompt_name,
    alias="champion",
    version=registered.version,
)

In [0]:
print(f"Registered {prompt_name} v{registered.version}; @champion -> v{registered.version}")

Registered genaicert.pawshield.policy_classifier_intent v9; @champion -> v9


In [0]:
# Read-only: load_prompt does not mutate state — it just resolves the alias.
loaded = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@champion")
print(f"Loaded back via alias; template length = {len(loaded.template)} chars")

Loaded back via alias; template length = 188 chars


## 3. Trace

`@mlflow.trace` on `_classify` (Phase 1) turns every classify call
into a span MLflow records automatically. One inference here, one
trace; production aggregates thousands. The trace UI is where you
debug "why did the agent return that?" without re-running anything.

Source: https://docs.databricks.com/aws/en/mlflow3/genai/tracing

In [0]:
sample_email = (
    "Hi PawShield — my dog Mochi had emergency surgery yesterday "
    "(invoice attached). When can I expect reimbursement? "
    "— Sarah Chen, CUST-CHEN-001"
)

In [0]:
# The @mlflow.trace decorator on _classify (Phase 1) records a span on
# every call — including when called transitively via predict().
response = agent.predict(ResponsesAgentRequest(
    input=[{"role": "user", "content": sample_email}]
))

Trace(trace_id=tr-4eb779cc906901fba54f9e7719714a6a)

In [0]:
# `response.output[0]` is an OutputItem Pydantic object (not a plain
# dict), so it doesn't support `["content"]` subscripting — use
# attribute access (`.content`) on the OutputItem. The inner content
# list IS a list of plain dicts, so `[0]["text"]` keeps dict access.
predicted_label = response.output[0].content[0]["text"]
print(f"Predicted intent: {predicted_label}")

# Source: https://mlflow.org/docs/latest/python_api/mlflow.html
# After predict() returns, the trace has closed, so get_active_trace_id()
# would return None. get_last_active_trace_id() returns the id of the
# most-recently-completed trace in this thread.
trace_id = mlflow.get_last_active_trace_id()
if trace_id:
    print(f"Trace id: {trace_id} (open in MLflow UI to inspect spans).")
else:
    print("No active trace captured (notebook may be outside a Databricks experiment).")

Predicted intent: complaint
Trace id: tr-4eb779cc906901fba54f9e7719714a6a (open in MLflow UI to inspect spans).


## 4. Evaluate

8 inline emails (2 per class), one custom `@scorer` for label
accuracy, plus the built-in `Correctness` and `Safety` judges so
the demo shows custom + built-in scorers composing in one call.

Sources:
* https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor
* https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers
* https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/judges

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers
# @scorer passes named args; declare only what you need; the leading `*` is
# defensive style, not required. Return scalar (int/float/bool/str) or
# mlflow.entities.Feedback.
from mlflow.genai.scorers import scorer, Correctness, Safety
import time


@scorer
def intent_accuracy(*, inputs, outputs, expectations) -> float:
    """1.0 if predicted intent matches expected, else 0.0; mean is accuracy."""
    predicted = ""
    if isinstance(outputs, dict):
        predicted = outputs.get("intent", "") or ""
    expected = (expectations or {}).get("expected_intent", "")
    return 1.0 if predicted.strip().lower() == expected.strip().lower() else 0.0


# Two examples per class — minimum for legible per-class accuracy in the
# MLflow UI. Grow this set as production monitoring surfaces edge cases.
#
# Each row's `expectations` carries TWO keys:
#   - `expected_intent`  — read by the custom `intent_accuracy` scorer.
#   - `expected_response` — read by the built-in `Correctness` scorer
#     (it's a hard-coded contract field name; without it Correctness
#     fails every row: `'correctness': N/N failed`).
# Same string in both keys; cheap insurance so both scorers run.
def _row(text, intent):
    return {"inputs": {"email_text": text},
            "expectations": {"expected_intent": intent,
                             "expected_response": intent}}

eval_records = [
    # --- claim ---
    _row("Submitting Mochi's ER bill from last night. Invoice attached.", "claim"),
    _row("Filing a new claim for Biscuit's dental cleaning ($340).", "claim"),
    # --- question ---
    _row("Does my Paw Plus tier cover annual vaccinations?", "question"),
    _row("How long is the waiting period for a new kitten?", "question"),
    # --- appeal ---
    _row("I'm appealing the denial on CLM-2026-03-00471 — the policy clearly covers it.", "appeal"),
    _row("My reimbursement was reduced. I already paid this year's deductible — please reconsider.", "appeal"),
    # --- complaint ---
    _row("Your portal has been broken for three days. This is unacceptable.", "complaint"),
    _row("I've been on hold for 40 minutes. Cancel my policy if no one answers.", "complaint"),
]


# Throttle + exponential-backoff retry on REQUEST_LIMIT_EXCEEDED — the
# 8 eval rows × 3 scorers (intent_accuracy, Correctness, Safety) fire
# 24+ FM calls quickly, which can trip the per-workspace QPS cap on
# pay-per-token endpoints. Mirrors the `_safe_invoke` pattern from c1301. For production-scale eval, graduate the
# judge endpoint to provisioned throughput instead.
THROTTLE_SECONDS = 0.3
MAX_RETRIES = 4


def _safe_predict(req):
    for attempt in range(MAX_RETRIES):
        try:
            out = agent.predict(req)
            time.sleep(THROTTLE_SECONDS)
            return out
        except Exception as e:
            msg = str(e)
            is_429 = "REQUEST_LIMIT_EXCEEDED" in msg or "429" in msg
            if not is_429 or attempt == MAX_RETRIES - 1:
                raise
            backoff = 2 ** (attempt + 1)
            print(f"  rate-limited; sleeping {backoff}s then retrying...")
            time.sleep(backoff)


def predict_fn(email_text: str):
    """Adapter from a row's `inputs` dict to the agent. Returns a flat dict
    so scorers can read `outputs['intent']` directly. Uses `_safe_predict`
    so the eval loop survives transient 429s without aborting the whole run."""
    resp = _safe_predict(ResponsesAgentRequest(
        input=[{"role": "user", "content": email_text}]
    ))
    # Same OutputItem .content access as the §3 smoke test — Pydantic
    # object, not dict; attribute lookup, then dict subscript on the
    # inner content list.
    return {"intent": resp.output[0].content[0]["text"]}

In [0]:
print(f"Eval set size: {len(eval_records)}")

Eval set size: 8


In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor
# evaluate(data, predict_fn, scorers) runs predict_fn on each row, applies
# scorers to (inputs, outputs, expectations, trace), records per-row scores
# plus aggregate metrics under an MLflow run.
with mlflow.start_run(run_name="policy_classifier_v1_seed_eval") as run:
    mlflow.genai.evaluate(
        data=eval_records,
        predict_fn=predict_fn,
        scorers=[intent_accuracy, Correctness(), Safety()],
    )
    print(f"Run ID: {run.info.run_id}")

2026/06/06 17:20:45 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/06 17:20:45 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]

Run ID: 5b921c6298c9496fbdbf6d2799163435


In [0]:
finished_run = mlflow.MlflowClient().get_run(run.info.run_id)
print("Headline metrics:")
for k, v in sorted(finished_run.data.metrics.items()):
    try:
        print(f"  {k}: {v:.3f}")
    except (TypeError, ValueError):
        print(f"  {k}: {v}")

Headline metrics:
  correctness/mean: 0.875
  intent_accuracy/mean: 0.625
  safety/mean: 1.000


## 5. Optimize (optional)

`mlflow.genai.optimize_prompts` takes the current `@champion` as a
seed, mutates it (rephrase, reorder, add/remove examples), scores
each variant against your eval set, and registers the best as a
new prompt version. You decide whether to promote it. The default
optimizer is GEPA.

The call below is commented out because each run costs ~10-30
minutes and many LLM calls; production optimization also wants 50+
eval rows (this notebook ships with 8) so signal beats noise.

Source: https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/automatically-optimize-prompts

In [0]:
# Source: https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/automatically-optimize-prompts
# Uncomment to run — costs ~10-30 minutes and many LLM calls.
#
# from mlflow.genai.optimize import GepaPromptOptimizer
# optimized = mlflow.genai.optimize_prompts(
#     predict_fn=predict_fn,
#     train_data=eval_records,
#     prompt_uris=[f"prompts:/{prompt_name}@champion"],
#     optimizer=GepaPromptOptimizer(),
#     scorers=[intent_accuracy],
# )
# mlflow.genai.set_prompt_alias(name=prompt_name, alias="champion",
#                               version=optimized.version)
print("Phase 5 — optimizer call commented out. See markdown above for what it would do.")

Phase 5 — optimizer call commented out. See markdown above for what it would do.


## 6. Deploy

**Code-based logging** — the artifact is the standalone
`policy_classifier_agent.py` workspace file (it calls
`mlflow.models.set_model(PolicyClassifierAgent())` at module
scope). `log_model(python_model="policy_classifier_agent.py",
registered_model_name=...)` lands it in UC Registry, ready for
serving. We stop short of actually creating the serving
endpoint (minutes to spin up, costs money); the c1001 notebook walks that
step end-to-end.

Why the agent isn't inline in this notebook: `python_model=<path>`
needs a real `.py` workspace **file**. Databricks workspace
**notebooks** uploaded with `--format SOURCE` have no `.py` on
disk, so the notebook source itself can't be the deployment
artifact — only a separately-uploaded `.py` workspace file can.

Sources:
* https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent
* https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle

In [0]:
# Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent
# Code-based logging: python_model=<path> points at the standalone
# `policy_classifier_agent.py` workspace file (the .py file calls
# `mlflow.models.set_model(PolicyClassifierAgent())` at module scope).
# `model_config` is the runtime parameterisation — load_context reads
# it; one class serves multiple deployments via different log_model
# invocations with different configs. `resources=[...]` declares what
# the endpoint needs to call; `registered_model_name` lands it in UC
# Registry in one step.
from mlflow.models.resources import DatabricksServingEndpoint

registered_model = f"{catalog}.{schema}.lifecycle_demo_agent"

In [0]:
with mlflow.start_run(run_name="policy_classifier_v1_deploy") as run:
    logged = mlflow.pyfunc.log_model(
        python_model="policy_classifier_agent.py",
        # `name=` is the current kwarg per MLflow 3 (May 2026). The older
        # `artifact_path=` synonym still works but emits a DeprecationWarning
        # on log_model. Use `name=` going forward.
        name="lifecycle_demo_agent",
        registered_model_name=registered_model,
        model_config={
            "prompt_uri":   f"prompts:/{prompt_name}@champion",
            "llm_endpoint": llm_endpoint,
        },
        resources=[DatabricksServingEndpoint(endpoint_name=llm_endpoint)],
        pip_requirements=["mlflow[databricks]", "databricks-sdk", "databricks-openai"],
    )
    print(f"Logged: {logged.model_uri}")
    print(f"Registered: {registered_model} v{logged.registered_model_version}")

🔗 View Logged Model at: https://<workspace>.cloud.databricks.com/ml/experiments/2638216843054092/models/m-feb6691bb6c04d02b0044f37d3041064?o=<WORKSPACE_ID>
2026/06/06 17:21:20 INFO mlflow.pyfunc: Predicting on input example to validate output
Registered model 'genaicert.pawshield.lifecycle_demo_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '9' of model 'genaicert.pawshield.lifecycle_demo_agent': https://<workspace>.cloud.databricks.com/explore/data/models/genaicert/pawshield/lifecycle_demo_agent/version/9?o=<WORKSPACE_ID>


Logged: models:/m-feb6691bb6c04d02b0044f37d3041064
Registered: genaicert.pawshield.lifecycle_demo_agent v9


In [0]:
# Source: https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent
# Next step (NOT run here): create a serving endpoint over the registered
# model. The c1001 notebook covers this end-to-end. A one-liner sketch:
#
#   from databricks.agents import deploy
#   deployment = deploy(
#       model_name=f"{catalog}.{schema}.lifecycle_demo_agent",
#       model_version=logged.registered_model_version,
#   )
print(f"Skipping serving-endpoint creation (see the c1001 deploy notebook). "
      f"To deploy: databricks.agents.deploy(model_name='{registered_model}', "
      f"model_version={logged.registered_model_version})")

Skipping serving-endpoint creation (see the c1001 deploy notebook). To deploy: databricks.agents.deploy(model_name='genaicert.pawshield.lifecycle_demo_agent', model_version=9)


## 7. Monitor (forward pointer)

Once the Phase 6 serving endpoint exists, **Agent Monitoring** (a.k.a.
Production Monitoring) reuses the same `@scorer` instances from
Phase 4 on live traffic — no second implementation, no parallel
harness. The `system.ai_gateway.usage` system table aggregates
per-call tokens / latency / errors across every endpoint in the
workspace, ready to join against per-trace scores for a
quality-vs-cost dashboard; the c1401 notebook builds it out.

## What you saw — the six pillars, recap

| # | Phase     | `mlflow.*` API(s) exercised                                          |
|---|-----------|----------------------------------------------------------------------|
| 1 | Author    | `mlflow.pyfunc.ResponsesAgent`                                       |
| 2 | Register  | `mlflow.genai.register_prompt`, `mlflow.genai.set_prompt_alias`, `mlflow.genai.load_prompt` |
| 3 | Trace     | `@mlflow.trace`, `mlflow.get_last_active_trace_id`                   |
| 4 | Evaluate  | `mlflow.genai.evaluate`, `@scorer`, `Correctness`, `Safety`          |
| 5 | Optimize  | `mlflow.genai.optimize_prompts` (with `GepaPromptOptimizer`)         |
| 6 | Deploy    | `mlflow.models.set_model`, `mlflow.pyfunc.log_model` (UC-registered) |
| + | Monitor   | (c1401 — Lakeview over `system.ai_gateway.usage` + traces)           |

The same six pillars structure every phase notebook. When
you hit a chapter notebook that focuses on one phase, this file is
the map back to where that phase fits in the overall loop.

## Sarah Chen's appeal — the full lifecycle thread, one request

The toy `PolicyClassifierAgent` above made the six-phase spine
visible in one file. This final section walks the CANONICAL
lifecycle artifact — Sarah Chen's appeal (A1) — through the
components the depth-chapter notebooks produced, so the
capstone closes on the case the rest of the book is built around.

The walk doesn't re-author any of these components — it invokes
them. Each `print` block names which notebook produced
the artifact being called.

In [0]:
# 1. A1 — Sarah's appeal email. Loaded from the Volume (created by the c0001 setup notebook).
sarah_email_path = (
    f"/Volumes/{catalog}/pawshield/bootstrap/"
    "claim_emails/sarah_chen_20260328_098473.eml"
)

In [0]:
with open(sarah_email_path) as f:
    sarah_email = f.read()

In [0]:
print(f"A1 (email): {len(sarah_email)} chars; first line:")
print(f"  {sarah_email.splitlines()[0]}")

A1 (email): 547 chars; first line:
  From: sarah.chen.austin@gmail.com


In [0]:
# 2. A5 — extracted intent (the prompt registered by c0301 as
#    claimclerk_extraction@champion, then the c0701 chain wraps it).
#    Here we show the extracted shape the chain would produce, since
#    rebuilding the chain inline duplicates the c0701 notebook.
#
# Mocked — live version runs in the c0301 / c0701 notebooks. We pin the
# expected output here so the lifecycle walk stays self-contained and
# deterministic (re-running this cell never hits an LLM).
extracted = {
    "intent": "appeal",
    "customer_id": "CUST-CHEN-001",
    "claim_id": "CLM-2026-03-00471",
    "pet_name": "Biscuit",
    "contact_phone": "512-555-0188",    # in-chain PII postprocess redacts on response (the chain's `_strip_pii_before_response` step); gateway guardrails don't apply to PyFunc
}

In [0]:
print(f"A5 (extracted): {extracted}")

A5 (extracted): {'intent': 'appeal', 'customer_id': 'CUST-CHEN-001', 'claim_id': 'CLM-2026-03-00471', 'pet_name': 'Biscuit', 'contact_phone': '512-555-0188'}


In [0]:
# 3. Retrieval — the PolicyPal Vector Search index `policy_chunks_index` built by c0801.
#    Filter by Sarah's state + tier (looked up from her customer record,
#    joined from A3's claim_id; the customer record itself isn't an
#    A-coded artifact — A1-A5 are email / policy PDF / claim row /
#    vet invoice PDF / prompt registry per c0001).
from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient(disable_notice=True)
policy_index = vs_client.get_index(
    endpoint_name="pawshield-vs",
    index_name=f"{catalog}.pawshield.policy_chunks_index",
)

In [0]:
top4 = policy_index.similarity_search(
    query_text="Why was my reimbursement only $512 on an $890 bill?",
    columns=["chunk_id", "section", "tier", "state"],
    num_results=4,
    filters={"state": "TX", "tier": "PawPlus"},
)
print("Retrieval:")
for row in top4["result"]["data_array"]:
    print(f"  {row}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Retrieval:
  ['pp_plus_tx_v3__chunk_006', '3.2 Reimbursement calculation', 'PawPlus', 'TX', 0.5368717974830858]
  ['pp_plus_tx_v3__chunk_001', '1.1 Definitions', 'PawPlus', 'TX', 0.4989713131611136]
  ['pp_plus_tx_v3__chunk_004', '2.4 Network status', 'PawPlus', 'TX', 0.49521117306359513]
  ['pp_plus_tx_v3__chunk_018', '4.7 Chronic condition exclusions and reimbursement handling', 'PawPlus', 'TX', 0.48247985989857056]


In [0]:
# 4. Agent reasoning — the c0901 ClaimClerk agent reads A5 (extracted
#    intent) + the claim history (joined from A3's claim_id) +
#    retrieved chunks (from A2's policy PDF, via `policy_chunks_index`)
#    and decides recommended_action. Inline here as a summary of
#    what the agent's predict() would return.
#
# Mocked — live version runs in the c0901 notebook. Inlined here so
# the capstone closes on the recommendation shape without depending on
# the multi-tool agent + MCP setup. trace_id uses the last completed
# trace (the predict() trace has already closed by this point).
agent_output = {
    "recommended_action": "approve_with_adjustment",
    "rationale": (
        "Per Section 4.7 of pp_plus_tx_v3, the $250 deductible is "
        "per Policy Year, not per claim. CLM-2026-01-00098 (Jan 2026, "
        "same chronic-condition cohort) already satisfied it. "
        "Re-applying the $250 deductible on CLM-2026-03-00471 left it "
        "$200 short of the $712 owed ($512 actual)."
    ),
    "citations": ["pp_plus_tx_v3 §4.7"],
    "trace_id": mlflow.get_last_active_trace_id(),
}

In [0]:
print(f"A-agent-output: {agent_output}")

A-agent-output: {'recommended_action': 'approve_with_adjustment', 'rationale': 'Per Section 4.7 of pp_plus_tx_v3, the $250 deductible is per Policy Year, not per claim. CLM-2026-01-00098 (Jan 2026, same chronic-condition cohort) already satisfied it. Re-applying the $250 deductible on CLM-2026-03-00471 left it $200 short of the $712 owed ($512 actual).', 'citations': ['pp_plus_tx_v3 §4.7'], 'trace_id': 'tr-7e197627a7f27adb4a849ec10a65a43c'}


In [0]:
# 5. Trace + Inference Table — the c1401 monitoring notebook closes the loop. The
#    inference-table row's `databricks_request_id` is its
#    Databricks-platform-issued primary key; the documented
#    MLflow-trace join key is `client_request_id` (caller-set
#    as a top-level JSON key in the model serving request body
#    per the inference-tables doc, exposed on the trace as
#    `trace.client_request_id` — the
#    documented `search_traces` filter field; `trace.request_id`
#    is explicitly NOT supported on Databricks-managed MLflow).
#    Compliance walks the join for an audit.
print(
    "Forensic walk:\n"
    "  - Inference Table: filter request_time::date = DATE'2026-03-28'\n"
    "    AND request:dataframe_records[0].question::string ILIKE '%CLM-2026-03-00471%'\n"
    "  - Lift the row's client_request_id; open the matching MLflow Trace via\n"
    "    MlflowClient().search_traces(filter_string=\"trace.client_request_id = '<id>'\")\n"
    "    (databricks_request_id is the inference-table primary key, not a\n"
    "     documented trace-side filter)\n"
    "  - Spans: embedding -> VS retrieval -> prompt render ->\n"
    "    LLM call -> in-chain PII postprocess (the chain's `_strip_pii_before_response` step)\n"
    "    (NB: AI Gateway PII Redaction is not available for custom\n"
    "     PyFunc endpoints — PII handling lives inside the chain.)"
)

Forensic walk:
  - Inference Table: filter request_time::date = DATE'2026-03-28'
    AND request:dataframe_records[0].question::string ILIKE '%CLM-2026-03-00471%'
  - Lift the row's client_request_id; open the matching MLflow Trace via
    MlflowClient().search_traces(filter_string="trace.client_request_id = '<id>'")
    (databricks_request_id is the inference-table primary key, not a
     documented trace-side filter)
  - Spans: embedding -> VS retrieval -> prompt render ->
    LLM call -> in-chain PII postprocess (the chain's `_strip_pii_before_response` step)
    (NB: AI Gateway PII Redaction is not available for custom
     PyFunc endpoints — PII handling lives inside the chain.)


That's the full lifecycle thread for one request — every
component the previous notebooks built, invoked in the order the
deployed system invokes them on a real customer email. The
six-phase spine above is the MLflow primitive layer; this walk is
the assembled system in action.